# HEALPix-Compress: Benchmarking and Visualization Notebook

**Objective**: Generate all benchmarks, tables, and figures for the TGRS submission of the HEALPix-Compress paper.

**Requirements**:
- Python 3.10+
- Required packages: `numpy`, `scipy`, `zarr`, `xarray`, `matplotlib`, `seaborn`, `pillow`, `opencv`, `healpix`, `tqdm`, `pandas`, `skimage`

**Data**: 
- Sentinel-2 L2A data (10m bands: B02, B03, B04, B08)
- Sentinel-1 GRD data (optional)

**Output**:
1. Benchmark tables (compression ratio, RMSE, PSNR, SSIM, etc.)
2. Figures for the paper (pipeline diagram, sensitivity analysis, etc.)
3. Geophysical metrics (NDVI error, classification accuracy)

**Note**: This notebook assumes you have access to Sentinel-2 data. For testing, we'll use synthetic data.

In [1]:
%load_ext autoreload
%autoreload 2

# NOTE: this used to only check `import pystac_client` -- if that package
# happened to already be installed (e.g. from a previous session) the whole
# block, including `pip install -U "zarr>=3"`, was silently skipped, leaving
# an old zarr (< 3, no `zarr.abc` module) in place and causing
# `ModuleNotFoundError: No module named 'zarr.abc'` when `import hcc` runs
# later. zarr's version is now checked independently.

def _deps_missing():
    try:
        import pystac_client  # noqa: F401
    except ImportError:
        return True
    try:
        import zarr
        if int(zarr.__version__.split('.')[0]) < 3:
            return True
    except ImportError:
        return True
    return False

try:
    import pywt
except:
    !pip install PyWavelets
    import pywt
try:
    import zfpy
except:
    !pip install zfpy
    import zfpy
try:
    import pysz
except:
    !pip install pysz
    import pysz
if _deps_missing():
    !pip install xarray-eopf  xdggs healpix-geo foscat pystac_client
    !pip install -U "zarr>=3"
    print("Installed/upgraded dependencies -- RESTART THE KERNEL now (a plain "
          "pip install does not replace an already-imported module in this "
          "running process), then re-run from the top.")

import zarr
print("zarr version:", zarr.__version__)
assert int(zarr.__version__.split('.')[0]) >= 3, (
    f"zarr {zarr.__version__} is active but zarr>=3 is required (hcc needs "
    "zarr.abc.buffer etc). If you just installed/upgraded zarr, restart the "
    "kernel -- pip install alone will not swap out an already-imported "
    "module."
)

  Using cached pywavelets-1.9.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
Using cached pywavelets-1.9.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (4.5 MB)
  Using cached zfpy-1.0.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (1.6 kB)
Using cached zfpy-1.0.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (982 kB)
  Using cached pysz-1.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (7.9 kB)
Using cached pysz-1.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.6 MB)
zarr version: 3.2.1


In [2]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from skimage.metrics import structural_similarity as ssim
from skimage import img_as_float
from scipy.ndimage import gaussian_filter
from scipy import fftpack
import zarr
from zarr.codecs import Blosc, Zstd
import warnings
warnings.filterwarnings('ignore')
import pystac_client
import xarray as xr
# Set random seed for reproducibility
np.random.seed(42)

## 0. get Sentinel-2 data

In [3]:
benchmark_coordinates = {
    "urban": {
        "location": "Paris, France",
        "wgs84": {"lat": 48.8566, "lon": 2.3522},
        "utm": {
            "zone": "31N",
            "x": 448212,  # Easting (m)
            "y": 5411935,  # Northing (m)
        },
        "sentinel2_tile": "T31TCJ",
        "recommended_date": "2025-05-01/2025-05-30",
        "cloud": 20,
        "notes": "Dense urban area with buildings, roads, and green spaces. High spectral variability.",
    },
    "water": {
        "location": "Lake Geneva, Switzerland/France",
        "wgs84": {"lat": 46.4980, "lon": 6.6323},
        "utm": {
            "zone": "32N",
            "x": 318500-700,
            "y": 5150200-3100,
        },
        "sentinel2_tile": "T32TMT",
        "cloud": 10,
        "recommended_date": "2025-05-01/2025-05-30",
        "notes": "Clear water body with low reflectance variability. Minimal cloud cover.",
    },
    "forest": {
        "location": "Black Forest, Germany",
        "wgs84": {"lat": 48.2667, "lon": 8.0167},
        "utm": {
            "zone": "32N",
            "x": 427200,
            "y": 5346500,
        },
        "sentinel2_tile": "T32TQM",
        "cloud": 20,
        "recommended_date": "2025-05-01/2025-08-01",
        "notes": "Temperate forest with dense coniferous and deciduous trees. Low cloud cover in summer.",
    },
    "agriculture": {
        "location": "Beauce, France (near Orléans)",
        "wgs84": {"lat": 48.0, "lon": 1.5},
        "utm": {
            "zone": "31N",
            "x": 375000,
            "y": 5319000,
        },
        "sentinel2_tile": "T31TFJ",
        "cloud": 20,
        "recommended_date": "2025-08-10/2025-10-10",
        "notes": "Intensive agriculture (wheat, corn, rapeseed). Large, homogeneous fields with sharp boundaries.",
    },
    "snow_ice": {
        "location": "Greenland (Nuuk Glacier)",
        "wgs84": {"lat": 72.5786, "lon": -38.4415},
        "utm": {
            "zone": "25X",  # Zone polaire nord
            "x": 321000,
            "y": 8055000,
        },
        "sentinel2_tile": "T25XVU",
        "cloud": 20,
        "recommended_date": "2025-03-10/2025-05-21",
        "notes": "Permanent snow/ice with high albedo. Stable surface for long-term analysis.",
    },
    "clouds": {
        "location": "Atlantic Ocean",
        "wgs84": {"lat": 40.0, "lon": -30.0},
        "utm": {
            "zone": "29N",
            "x": 166021,
            "y": 4426413+3000,
        },
        "sentinel2_tile": "T29SQD",
        "cloud": 90,
        "recommended_date": "2025-04-10/2025-05-30",
        "notes": "Frequent cloud cover. Used to test robustness to cloud masks and high-reflectance values.",
    },
    "ocean": {
        "location": "Celtic Sea (off Brittany, France)",
        "wgs84": {"lat": 48.0, "lon": -5.0},
        "utm": {
            "zone": "30N",
            "x": 300000,
            "y": 5318000,
        },
        "cloud": 10,
        "sentinel2_tile": "T30TWM",
        "recommended_date": "2025-01-15/2025-05-10",
        "notes": "Deep oceanic water with very low reflectance. Ideal for testing compression of homogeneous low-signal areas.",
    },
}

In [4]:
import pyproj

def _add_latlon(ds: xr.Dataset, transformer: pyproj.Transformer) -> xr.Dataset:
    """Attach latitude/longitude coords + CF metadata to a Dataset with (x,y)."""
    if not {"x", "y"}.issubset(ds.dims):
        return ds

    xx, yy = np.meshgrid(ds["x"].values, ds["y"].values, indexing="xy")
    lon, lat = transformer.transform(xx, yy)

    ds = ds.assign_coords(
        longitude=(("y", "x"), lon),
        latitude=(("y", "x"), lat),
    )
    ds["latitude"].attrs.update(
        {
            "standard_name": "latitude",
            "long_name": "Latitude",
            "units": "degrees_north",
            "axis": "Y",
        }
    )
    ds["longitude"].attrs.update(
        {
            "standard_name": "longitude",
            "long_name": "Longitude",
            "units": "degrees_east",
            "axis": "X",
        }
    )

    # Make sure vars with (y,x) advertise the aux coords
    for var in ds.data_vars:
        if {"y", "x"}.issubset(ds[var].dims):
            existing = ds[var].attrs.get("coordinates", "").split()
            ds[var].attrs["coordinates"] = " ".join(
                sorted(set(existing) | {"latitude", "longitude"})
            )
    return ds


def add_latlon(
    path: str, ds: xr.Dataset, transformer: pyproj.Transformer
) -> xr.Dataset:
    """Wrapper for safe application on a node dataset."""
    if ds is None:
        print(path, "no dataset")
        return ds
    if not {"x", "y"}.issubset(ds.dims):
        print(path, "not both x,y")
        return ds
    return _add_latlon(ds, transformer)


def add_latlon_to_dt(
    dt: xr.DataTree,
    paths: list[str] | None = None,
) -> xr.DataTree:
    """Return a new DataTree with latitude/longitude coords added on selected paths."""

    try:
        crs_code = dt.attrs["other_metadata"]["horizontal_CRS_code"]

        src_crs = pyproj.CRS.from_string(crs_code)
    except:
        crs_code = dt.attrs["other_metadata"]["horizontal_crs_code"]

        src_crs = pyproj.CRS.from_string(crs_code)
        

    transformer = pyproj.Transformer.from_crs(
        src_crs,
        pyproj.CRS.from_epsg(4326),
        always_xy=True,
    )

    if paths is None:
        items = dt.subtree_with_keys
    else:
        items = [(p, dt[p]) for p in paths]

    out_dict = {}

    for p, node in items:
        if isinstance(node, xr.DataTree):
            obj = node.ds

        elif isinstance(node, xr.DataArray):
            obj = node.to_dataset(name=node.name or p.split("/")[-1])

        elif isinstance(node, xr.Dataset):
            obj = node

        else:
            raise TypeError(
                f"Unsupported object at path {p!r}: {type(node)}"
            )

        out_dict[p] = add_latlon(p, obj, transformer)

    out = xr.DataTree.from_dict(out_dict)
    out.attrs = dt.attrs.copy()

    return out
    

In [5]:


def extract_bench_data(LON,LAT,UTMX,UTMY,DATE,img_type,cloud_cover,force=False,plot=False,xsize=1512):

    try: 
        name='data/%s_data.zarr'%(domain)
        if force:
            name=name+'__8888'
        dt = xr.open_datatree(
            name,
            engine="zarr",
            consolidated=False,   
            chunks={}            # place the dask chunk as is in zarr chunk
        )
        print('data/%s_data.zarr already done'%(domain))
    except:
        xr.set_options(keep_attrs=True, display_expand_attrs=False)
        # Access cloud-optimized Sentinel-2 data via the EOPF STAC catalog
        catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu")
        
        # Search criteria optimized for wave analysis
        collection = "sentinel-2-l2a"
        items = list(
            catalog.search(
                datetime=DATE,
                collections=[collection],
                intersects=dict(type="Point", coordinates=[LON, LAT]),
                
                query={
                    "eo:cloud_cover": {
                        "lt": cloud_cover
                    },  # Cloud cover < 20% ensures clear ocean surface
                    "view:sun_elevation": {
                        "gt": 25
                    },  # Filter for high sun elevation > 25° (→ sun zenith angle < 65°),
                    # which places the sun near the zenith.
                },
            ).items()
        )
        
        for item in items:
            print(f"✅ {item.id}")
        
        item = items[0]
    
        dt = xr.open_datatree(
            item.assets["product"].href, engine="zarr", chunks={}, decode_timedelta=True
        )
        
        if plot:
            dt["measurements"]["reflectance"]["r60m"]["b01"].plot()
        else:
            latlon_dt = add_latlon_to_dt(dt,paths=['measurements/reflectance/r10m/b04'])
            latlon_dt = latlon_dt["measurements"]["reflectance"]["r10m"]["b04"]
        
            ix=np.argmin((latlon_dt.x.values-UTMX)**2)
            iy=np.argmin((latlon_dt.y.values-UTMY)**2)
            if ix<xsize//2:
                ix=xsize//2
            if iy<xsize//2:
                iy=xsize//2
            if ix>latlon_dt.x.shape[0]-xsize//2:
                ix=latlon_dt.x.shape[0]-xsize//2
            if iy>latlon_dt.y.shape[0]-xsize//2:
                iy=latlon_dt.y.shape[0]-xsize//2
            print(ix,iy)
            val=latlon_dt.sel(x=slice(latlon_dt.x[ix-xsize//2],latlon_dt.x[ix+xsize//2-1]),
                              y=slice(latlon_dt.y[iy-xsize//2],latlon_dt.y[iy+xsize//2-1]))

            val.to_zarr('data/%s_data.zarr'%(img_type), mode="w",
                zarr_version=2,align_chunks=True)
            print('data/%s_data.zarr SAVED'%(img_type))

In [6]:
for domain in benchmark_coordinates:
    extract_bench_data(
        benchmark_coordinates[domain]["wgs84"]["lon"],benchmark_coordinates[domain]["wgs84"]["lat"],
        benchmark_coordinates[domain]["utm"]["x"],benchmark_coordinates[domain]["utm"]["y"],
        benchmark_coordinates[domain]["recommended_date"],domain,benchmark_coordinates[domain]["cloud"],force=False)

data/urban_data.zarr already done
data/water_data.zarr already done
data/forest_data.zarr already done
data/agriculture_data.zarr already done
data/snow_ice_data.zarr already done
data/clouds_data.zarr already done
data/ocean_data.zarr already done


# compute HEALPIX patch

In [7]:
from skimage import exposure
from scipy.interpolate import griddata
import healpix_geo
# REPLACED BilinearResampler with scipy.interpolate.griddata (method='cubic')
# (see chat): BilinearResampler's IDW-1 kernel uses sigma_m tied to the
# HEALPix pixel scale (~6.2 m at level 20) rather than the coarser native
# UTM pixel spacing (10 m). Since that kernel is narrower than the source
# sampling interval, it does not act as a proper anti-aliasing filter, and
# was found (2D power-spectrum diagnostic, see chat) to inject spurious
# high-frequency content into the HEALPix tile that is not present in the
# native UTM data or in a griddata-cubic-interpolated tile. Switching to
# griddata(..., method='cubic') removed the artifact -- confirmed by a
# clean, matching power spectrum. This is now the reference resampling
# method for the HEALPix side of the benchmark.
from foscat.Plot import lgnomproject
from healpix_resample import CloughTocherResampler as Resampler

level=20

SIZE=1024

try:
    c = np.load('data/all_image_cell.npy')
    v = np.load('data/all_image_data.npy')
except:
    k=0
    c=np.zeros([len(benchmark_coordinates),SIZE*SIZE])
    v=np.zeros([len(benchmark_coordinates),SIZE*SIZE])

    N_CHILDREN = SIZE * SIZE  # 4**(level - 10): children of one level-10 base tile

    for domain in benchmark_coordinates:

        dt = xr.open_datatree(
            'data/%s_data.zarr'%(domain),
            engine="zarr",
            consolidated=False,
            chunks={}            # place the dask chunk as is in zarr chunk
        )
        img=dt['b04'].values
        img[np.isnan(img)]=np.nanmean(img)
        lon = dt["longitude"].values.astype(np.float64).flatten()
        lat = dt["latitude"].values.astype(np.float64).flatten()

        # 1. Which level-10 base tile do most of the UTM samples fall into?
        #    (same "dominant tile" logic as before, now computed directly
        #    from lonlat_to_healpix instead of the resampler's own cell_ids)
        cover_ids = healpix_geo.nested.lonlat_to_healpix(lon, lat, level,ellipsoid="WGS84").astype(np.int64)
        parent_ids = cover_ids // N_CHILDREN
        n, r = np.unique(parent_ids, return_inverse=True)
        print('Cover %.2f%%'%(100*np.max(np.bincount(r))/N_CHILDREN),np.argmax(np.bincount(r)))
        
        dominant_parent = n[np.argmax(np.bincount(r))]

        # 2. Full, guaranteed-complete set of the dominant tile's 1024x1024
        #    level-20 children (nested indexing: children of a level-10
        #    parent are the contiguous block [parent*N_CHILDREN, (parent+1)*N_CHILDREN) ).
        cell_ids = dominant_parent * N_CHILDREN + np.arange(N_CHILDREN, dtype=np.int64)

        resampler_op = Resampler(lon,lat,level,out_cell_ids=cell_ids)

        res = resampler_op.resample(img.flatten())
        cell_data, cell_ids = res.cell_data, res.cell_ids
        
        c[k] = cell_ids
        v[k] = cell_data

        k=k+1
        
    np.save('data/all_image_cell.npy',c)
    np.save('data/all_image_data.npy',v)
plt.figure(figsize=(16,8))
for k in range(c.shape[0]):
    lgnomproject(c[k],exposure.equalize_hist(v[k]),2**(level),hold=False,fov_deg=0.1,interp=True,sub=(2,4,1+k),cbar=True,cmap='grey')

ModuleNotFoundError: No module named 'foscat'

In [ ]:
from hcc.compression.zorder import expand_tile
from healpix_analyse.ps import ps

for k in range(7):
    nu,psd=ps(expand_tile(v[k]))
    plt.loglog(nu,psd)